In [7]:
import json
import os


CHECKPOINTS = [
    "results_pretrain/base_rope_42",
    "results_pretrain/base_nope_w128d16r0h1g1_42",
]


def load_results(checkpoint: str) -> dict:
    path = os.path.join(checkpoint, "pretrain_eval.txt")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data


def extract_metrics(stats: dict) -> dict:
    out = {}
    for k, v in stats.items():
        if not k.endswith(",none"):
            continue
        name = k[:-5]
        if name.endswith("_stderr"):
            continue
        out[name] = v
    return out


def fmt(v) -> str:
    if v is None:
        return "-"
    return f"{v:.4f}"


def fmt_with_delta(cur, base) -> str:
    if cur is None:
        return "-"
    if base is None:
        return fmt(cur)
    return f"{cur:.4f}({cur - base:+.4f})"


def print_markdown_table(all_results: dict, checkpoints: list):
    datasets = []
    metrics_by_ds = {}
    for ckpt in checkpoints:
        for ds, stats in all_results.get(ckpt, {}).items():
            if ds not in metrics_by_ds:
                datasets.append(ds)
                metrics_by_ds[ds] = []
            for m in extract_metrics(stats).keys():
                if m not in metrics_by_ds[ds]:
                    metrics_by_ds[ds].append(m)

    names = [os.path.basename(c.rstrip("/")) for c in checkpoints]
    header = ["Dataset", "Metric"] + names

    lines = []
    lines.append("| " + " | ".join(header) + " |")
    lines.append("|" + "|".join([":---:"] * len(header)) + "|")

    for ds in datasets:
        for m in metrics_by_ds[ds]:
            base_metrics = extract_metrics(all_results.get(checkpoints[0], {}).get(ds, {}))
            base_val = base_metrics.get(m)

            cells = [ds, m, fmt(base_val)]
            for ckpt in checkpoints[1:]:
                cur_metrics = extract_metrics(all_results.get(ckpt, {}).get(ds, {}))
                cur_val = cur_metrics.get(m)
                cells.append(fmt_with_delta(cur_val, base_val))
            lines.append("| " + " | ".join(cells) + " |")

    print("\n".join(lines))


def main():
    all_results = {}
    for checkpoint in CHECKPOINTS:
        all_results[checkpoint] = load_results(checkpoint)

    print_markdown_table(all_results, CHECKPOINTS)


if __name__ == "__main__":
    main()

| Dataset | Metric | base_rope_42 | base_nope_w128d16r0h1g1_42 |
|:---:|:---:|:---:|:---:|
| wikitext | word_perplexity | 46.5636 | 42.0498(-4.5138) |
| wikitext | byte_perplexity | 2.0508 | 2.0121(-0.0387) |
| wikitext | bits_per_byte | 1.0362 | 1.0087(-0.0275) |
| lambada_openai | perplexity | 118.1028 | 88.4866(-29.6162) |
| lambada_openai | acc | 0.2507 | 0.2604(+0.0097) |
| hellaswag | acc | 0.2894 | 0.2949(+0.0055) |
| hellaswag | acc_norm | 0.3148 | 0.3272(+0.0124) |
| xstorycloze_en | acc | 0.5460 | 0.5526(+0.0066) |
| piqa | acc | 0.6251 | 0.6338(+0.0087) |
| piqa | acc_norm | 0.6208 | 0.6235(+0.0027) |
| arc_easy | acc | 0.5114 | 0.5421(+0.0307) |
| arc_easy | acc_norm | 0.4487 | 0.4701(+0.0215) |
| winogrande | acc | 0.5091 | 0.5193(+0.0103) |
| boolq | acc | 0.5676 | 0.5893(+0.0217) |
| truthfulqa_mc1 | acc | 0.2399 | 0.2375(-0.0024) |
